# pctrl：最小化 HV / VH / DA / AD 四组 basis 的双光子计数

这个示例在 Jupyter 中直接调用 `qtools.pctrl`。每次优化评估会：

1. 移动 Alice 和 Bob 的 basis 电机；
2. 在 `HV`、`VH`、`DA`、`AD` 四组 basis 下读取 TDC coincidence rate；
3. 将四组 rate 的平均值作为目标，用 Nelder-Mead 调整 pctrl 指定 channel 的 QWP-HWP-QWP 波片角度。

电机连接参数从独立的 `motor_config.yaml` 读取；basis 校准角度从 `polmeas_config.yaml` 读取，若 `motor_config.yaml` 中也有 `bases` 字段则优先使用它。请在实验目录中运行 notebook，并先完成 pctrl、motor 和 pmeas 的配置。

## 1. 导入依赖并设置实验参数

下面的 TDC 参数必须和当前 coincidence 窗口一致。`TARGET` 是要由 pctrl 优化的 channel；如果要优化另一侧，改成对应的 optical path。

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from qtools.motor import MotorConfigManager
from qtools.pctrl import PolarizationController
from qtools.pmeas.config import ConfigManager as PolmeasConfigManager
from qtools.pmeas.entanglement import _get_basis_angle
from qtools.pmeas.motor_manager import MotorManager

# 在实验目录启动 Jupyter，或把下面三个路径改成绝对路径。
PCTRL_CONFIG = Path.cwd() / "pctrl_config.yaml"
MOTOR_CONFIG = Path.cwd() / "motor_config.yaml"
POLMEAS_CONFIG = Path.cwd() / "polmeas_config.yaml"

# pctrl 要优化的 channel。也可以改成 "bob"。
TARGET = "alice"

# motor_config.yaml 中的设备 target；常见配置直接使用 optical_path。
MOTOR_TARGETS = {"alice": "alice", "bob": "bob"}

BASIS_PAIRS = (("H", "V"), ("V", "H"), ("D", "A"), ("A", "D"))
MEASURE_DURATION = 1.0       # s per TDC reading
MEASURE_SAMPLES = 3
CH_START = 1                 # TDC channel numbers are 1-based
CH_STOP = 4
DELAY = 100                   # coincidence delay, TDC bins
WINDOW_START = 109            # coincidence window start, TDC bins
WINDOW_STOP = 110             # coincidence window stop, TDC bins
THRESHVOLT = 0.6              # TDC threshold, V

MAXITER = 40
ANGLE_TOL = 1.0
STEP_SIZE_DEG = 15.0
SPEED_DEG_S = 10.0
SETTLE_TIME = 0.2


In [ ]:
for config_path in (PCTRL_CONFIG, MOTOR_CONFIG, POLMEAS_CONFIG):
    print(f"{config_path}: {'OK' if config_path.exists() else 'missing'}")

if not PCTRL_CONFIG.exists():
    raise FileNotFoundError(
        f"pctrl config not found: {PCTRL_CONFIG}. Set PCTRL_CONFIG to your experiment config."
    )
if not MOTOR_CONFIG.exists():
    raise FileNotFoundError(
        f"motor config not found: {MOTOR_CONFIG}. Set MOTOR_CONFIG to your motor_config.yaml."
    )
if not POLMEAS_CONFIG.exists():
    raise FileNotFoundError(
        f"basis config not found: {POLMEAS_CONFIG}. Set POLMEAS_CONFIG to your polmeas_config.yaml."
    )


## 2. 连接 pctrl 和 basis 电机

`MotorManager` 使用 `motor_config.yaml` 建立电机连接；`polmeas_config.yaml` 只用于读取 Alice/Bob 的 basis 校准角度。

In [ ]:
pctrl = PolarizationController(config_path=str(PCTRL_CONFIG))
motor_cfg = MotorConfigManager.resolve(MOTOR_CONFIG)
polmeas_cfg = PolmeasConfigManager.resolve(POLMEAS_CONFIG)
motor_mgr = MotorManager(motor_cfg)

basis_stages = {
    side: motor_mgr.get_motor(MOTOR_TARGETS[side])
    for side in ("alice", "bob")
}
print("pctrl and basis motors are ready")


## 3. 定义 basis 切换和四组 coincidence 计数

basis 角度优先从 `motor_config.yaml` 中对应设备的 `bases` 字段读取；没有时回退到 `polmeas_config.yaml`。这允许电机连接参数和光学 basis 校准参数分别维护。

In [ ]:
def _entry(cfg, target):
    try:
        return cfg.get_device(target) or {}
    except (AttributeError, ValueError):
        return {}


def basis_angle(side, basis):
    basis = str(basis).upper()
    # Standalone motor config may include calibrated basis angles.
    for cfg, target in ((motor_cfg, MOTOR_TARGETS[side]), (polmeas_cfg, side)):
        entry = _entry(cfg, target)
        bases = entry.get("bases", {})
        if isinstance(bases, dict) and basis in bases:
            return float(bases[basis])
    return float(_get_basis_angle(polmeas_cfg, side, basis))


def move_basis_pair(pair):
    """Move both analyzer motors to a pair such as ("H", "V")."""
    for side, basis in zip(("alice", "bob"), pair):
        basis_stages[side].move_to(basis_angle(side, basis))
    for stage in basis_stages.values():
        stage.wait_move()


def read_coincidence(meas_mgr):
    result = meas_mgr.measure(
        duration=MEASURE_DURATION,
        samples=MEASURE_SAMPLES,
        measure_type="pairs",
        ch_start=CH_START,
        ch_stop=CH_STOP,
        delay=DELAY,
        window_start=WINDOW_START,
        window_stop=WINDOW_STOP,
        threshvolt=THRESHVOLT,
    )
    mean, std = result[0]
    return float(mean), float(std)


class FourBasisCoincidenceMetric:
    """Callable pctrl metric that minimizes the four basis rates."""

    def __init__(self):
        self.history = []

    def measure_all(self, meas_mgr):
        counts = {}
        errors = {}
        for pair in BASIS_PAIRS:
            label = "".join(pair)
            move_basis_pair(pair)
            counts[label], errors[label] = read_coincidence(meas_mgr)
        objective = float(np.mean(list(counts.values())))
        entry = {
            "counts": counts,
            "errors": errors,
            "objective": objective,
        }
        self.history.append(entry)
        return entry

    def __call__(self, meas_mgr):
        return self.measure_all(meas_mgr)["objective"]


metric = FourBasisCoincidenceMetric()


## 4. 优化前测量

先记录当前波片角度下四组 basis 的 coincidence rate，便于和优化结果比较。

In [ ]:
def show_counts(title, measurement):
    print(title)
    for pair in BASIS_PAIRS:
        label = "".join(pair)
        print(f"  {label}: {measurement['counts'][label]:.3f} +/- {measurement['errors'][label]:.3f} cps")
    print(f"  mean: {measurement['objective']:.3f} cps")

baseline = metric.measure_all(pctrl.meas_mgr)
show_counts("Baseline", baseline)


## 5. 运行 pctrl 优化

`metric_fn` 每次被调用时都会重新测量四组 basis。`minimize=True` 让 pctrl 的 Nelder-Mead 优化器寻找更低的四组平均 coincidence rate。

In [ ]:
def progress(iteration, angles, value, _raw_measurement):
    # The metric entry immediately before this callback belongs to this evaluation.
    entry = metric.history[-1]
    entry["evaluation"] = int(iteration)
    entry["angles"] = [float(angle) for angle in angles]
    counts = " ".join(
        f"{label}={entry['counts'][label]:.2f}"
        for label in ("HV", "VH", "DA", "AD")
    )
    print(f"eval {iteration:03d} | mean={value:.3f} cps | {counts}")

result = pctrl.optimize(
    target=TARGET,
    metric_fn=metric,
    minimize=True,
    method="Nelder-Mead",
    maxiter=MAXITER,
    tol=ANGLE_TOL,
    step_size_deg=STEP_SIZE_DEG,
    speed_deg_s=SPEED_DEG_S,
    settle_time=SETTLE_TIME,
    duration=MEASURE_DURATION,
    samples=MEASURE_SAMPLES,
    measure_type="pairs",
    ch_start=CH_START,
    ch_stop=CH_STOP,
    delay=DELAY,
    window_start=WINDOW_START,
    window_stop=WINDOW_STOP,
    threshvolt=THRESHVOLT,
    callback=progress,
    save_config=False,
    verbose=True,
)

print(result.message)
print("Optimal pctrl angles:", result.optimal_angles)


## 6. 优化后测量和收敛曲线

最后重新读取四组 basis，避免把某一次带噪声的评估直接当作最终结果。

In [ ]:
final = metric.measure_all(pctrl.meas_mgr)
final["evaluation"] = "final"
final["angles"] = [float(angle) for angle in result.optimal_angles]
show_counts("After optimization", final)

numeric_history = [
    entry for entry in metric.history
    if isinstance(entry.get("evaluation"), int)
]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(
    [entry["evaluation"] for entry in numeric_history],
    [entry["objective"] for entry in numeric_history],
    marker="o",
)
axes[0].set(xlabel="evaluation", ylabel="mean coincidence rate (cps)", title="Optimization objective")
axes[0].grid(True, alpha=0.3)

for label in ("HV", "VH", "DA", "AD"):
    axes[1].plot(
        [entry["evaluation"] for entry in numeric_history],
        [entry["counts"][label] for entry in numeric_history],
        marker="o",
        label=label,
    )
axes[1].set(xlabel="evaluation", ylabel="coincidence rate (cps)", title="Four basis rates")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()


## 7. 导出评估历史（可选）

CSV 包含每次评估的波片角度、四组 rate、标准差和目标值；不会写回 pctrl 配置。

In [ ]:
CSV_PATH = Path.cwd() / "pctrl_two_photon_basis_minimization.csv"
fieldnames = ["evaluation", "objective", "angle_1", "angle_2", "angle_3"]
for label in ("HV", "VH", "DA", "AD"):
    fieldnames.extend([f"{label}_rate", f"{label}_std"])

with CSV_PATH.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    for entry in metric.history:
        angles = entry.get("angles", [])
        row = {
            "evaluation": entry.get("evaluation", ""),
            "objective": entry["objective"],
            "angle_1": angles[0] if len(angles) > 0 else "",
            "angle_2": angles[1] if len(angles) > 1 else "",
            "angle_3": angles[2] if len(angles) > 2 else "",
        }
        for label in ("HV", "VH", "DA", "AD"):
            row[f"{label}_rate"] = entry["counts"][label]
            row[f"{label}_std"] = entry["errors"][label]
        writer.writerow(row)

print(f"saved {CSV_PATH}")


## 8. 关闭硬件连接

完成实验或发生异常后执行这一单元。

In [ ]:
try:
    pctrl.close()
finally:
    motor_mgr.disconnect_all()
print("hardware connections closed")
